# Notebook 07 — Ward-Level Intelligence

## Roadmap
Phase 7 — Spatial / Ward-Level Intelligence.

## Objective
Build a consolidated profile for each source ward identifier,
combining demand, booking outcomes, exploratory cluster assignments,
and anomaly rankings.

## Scope
Validate joins across existing outputs and produce readable ward
comparison tables. Reuse completed modelling rather than refitting it.

## Geographic limitation
The inspected data provides no verified link from source identifiers
to ward names, coordinates, or boundaries. Geographic mapping and
spatial relationships remain pending.

Do not infer geography from identifier numbers. Keep w_0 explicitly
flagged because its geographic meaning remains unresolved.

## Interpretation
Profiles support investigation. Cluster assignments and anomaly scores
do not establish causes or justify automatic operational interventions.

## 7.1 — Load Existing Outputs and Validate Identifier Coverage

Load ward features, exploratory cluster assignments, and anomaly rankings.

Check identifier uniqueness, coverage, and consistency before joining.
Preserve all source records, including those excluded from model fitting.

In [1]:
from pathlib import Path
import subprocess
import pandas as pd

PROJECT_DIR = Path("/content/NammaFlow")

if not PROJECT_DIR.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/Narasimhan15/NammaFlow.git",
            str(PROJECT_DIR),
        ],
        check=True,
    )

INPUT_PATHS = {
    "features": PROJECT_DIR / "data/features/ward_intelligence.csv",
    "clusters": PROJECT_DIR / "reports/ward_clustering/ward_assignments.csv",
    "anomalies": (
        PROJECT_DIR / "reports/anomaly_detection/ward_anomaly_rankings.csv"
    ),
    "exclusions": (
        PROJECT_DIR / "reports/anomaly_detection/excluded_records.csv"
    ),
}

tables = {}

for name, path in INPUT_PATHS.items():
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing {name}: {path}. "
            "If this runtime has an older repository checkout, "
            "we need to refresh it before continuing."
        )

    table = pd.read_csv(path)

    assert "ward_num" in table.columns, f"{name}: missing ward_num"
    assert table["ward_num"].notna().all(), f"{name}: missing identifiers"
    assert table["ward_num"].is_unique, f"{name}: duplicate identifiers"

    tables[name] = table

features = tables["features"]
clusters = tables["clusters"]
anomalies = tables["anomalies"]
exclusions = tables["exclusions"]

feature_ids = set(features["ward_num"])
cluster_ids = set(clusters["ward_num"])
anomaly_ids = set(anomalies["ward_num"])
excluded_ids = set(exclusions["ward_num"])

for name, table in tables.items():
    unknown = set(table["ward_num"]) - feature_ids
    assert not unknown, f"{name}: unknown identifiers {sorted(unknown)}"

assert anomaly_ids.isdisjoint(excluded_ids), \
    "Anomaly rankings overlap with exclusions."

assert anomaly_ids | excluded_ids == feature_ids, \
    "Anomaly rankings and exclusions do not cover all feature records."

assert cluster_ids == anomaly_ids, \
    "Cluster and anomaly eligibility differ; investigate before joining."

print("Input coverage:")
display(pd.DataFrame([
    {
        "input": name,
        "rows": len(table),
        "unique_ids": table["ward_num"].nunique(),
        "includes_w_0": table["ward_num"].eq("w_0").any(),
    }
    for name, table in tables.items()
]))

print("\nRecords without fitted cluster/anomaly results:")
print(sorted(excluded_ids))

print("\nRequired output fields:")
for name, required in {
    "clusters": ["cluster"],
    "anomalies": ["anomaly_score", "review_position", "low_booking_count"],
    "exclusions": ["exclusion_reason"],
}.items():
    missing = set(required) - set(tables[name].columns)
    assert not missing, f"{name}: missing fields {sorted(missing)}"
    print(f"{name}: OK")

print("\nIdentifier coverage checks passed.")

Input coverage:


,input,rows,unique_ids,includes_w_0
0,features,244,244,True
1,clusters,241,241,True
2,anomalies,241,241,True
3,exclusions,3,3,False



Records without fitted cluster/anomaly results:
['b_168', 'b_26', 'b_44']

Required output fields:
clusters: OK
anomalies: OK
exclusions: OK

Identifier coverage checks passed.


## 7.2 — Build Consolidated Ward Profiles

Verify that shared inputs match across exported results before joining.

Preserve all 244 source records. Keep unavailable model outputs missing,
retain exclusion reasons, and label clusters as exploratory.

Geographic mapping remains unverified for every identifier.

In [2]:
import numpy as np

# Check that exported model inputs match the current feature table.
CONSISTENCY_COLS = [
    "ward_searches",
    "ward_search_to_booking_rate",
    "ward_booking_completion_rate",
]

feature_lookup = features.set_index("ward_num")

for name, table in {
    "clusters": clusters,
    "anomalies": anomalies,
}.items():
    exported = table.set_index("ward_num")

    for column in CONSISTENCY_COLS:
        assert column in exported.columns, f"{name}: missing {column}"

        np.testing.assert_allclose(
            exported[column].to_numpy(),
            feature_lookup.loc[exported.index, column].to_numpy(),
            rtol=1e-6,
            atol=1e-8,
            equal_nan=True,
            err_msg=f"{name}: input mismatch in {column}",
        )

# Select model outputs explicitly to avoid duplicated feature columns.
ward_profiles = (
    features
    .merge(
        clusters[["ward_num", "cluster"]].rename(
            columns={"cluster": "exploratory_cluster"}
        ),
        on="ward_num",
        how="left",
        validate="one_to_one",
    )
    .merge(
        anomalies[["ward_num", "anomaly_score", "review_position"]],
        on="ward_num",
        how="left",
        validate="one_to_one",
    )
    .merge(
        exclusions[["ward_num", "exclusion_reason"]],
        on="ward_num",
        how="left",
        validate="one_to_one",
    )
)

ward_profiles["exploratory_cluster"] = (
    ward_profiles["exploratory_cluster"].astype("Int64")
)
ward_profiles["review_position"] = (
    ward_profiles["review_position"].astype("Int64")
)

ward_profiles["has_model_results"] = (
    ward_profiles["ward_num"].isin(anomaly_ids)
)
ward_profiles["low_booking_count"] = ward_profiles["ward_bookings"].lt(10)
ward_profiles["is_w_0"] = ward_profiles["ward_num"].eq("w_0")
ward_profiles["geographic_mapping_status"] = "unverified"

assert len(ward_profiles) == len(features)
assert ward_profiles["ward_num"].is_unique
assert set(ward_profiles["ward_num"]) == feature_ids

unmodelled = ~ward_profiles["has_model_results"]
model_cols = [
    "exploratory_cluster", "anomaly_score", "review_position"
]

assert ward_profiles.loc[unmodelled, model_cols].isna().all().all()
assert ward_profiles.loc[~unmodelled, model_cols].notna().all().all()
assert ward_profiles["exclusion_reason"].notna().equals(unmodelled)

print("Shared input values match.")
print(f"Consolidated profiles: {len(ward_profiles)}")
print(f"With model results: {ward_profiles['has_model_results'].sum()}")
print(f"Without model results: {unmodelled.sum()}")

PREVIEW_COLS = [
    "ward_num",
    "ward_searches",
    "ward_bookings",
    "ward_search_to_booking_rate",
    "ward_booking_completion_rate",
    "exploratory_cluster",
    "anomaly_score",
    "review_position",
    "has_model_results",
    "low_booking_count",
    "exclusion_reason",
]

print("\nSelected profiles, including exclusions:")
display(
    ward_profiles.loc[
        ward_profiles["ward_num"].isin(
            ["b_181", "b_143", "w_0", "b_168", "b_26", "b_44"]
        ),
        PREVIEW_COLS,
    ].round(4)
)

Shared input values match.
Consolidated profiles: 244
With model results: 241
Without model results: 3

Selected profiles, including exclusions:


,ward_num,ward_searches,ward_bookings,ward_search_to_booking_rate,ward_booking_completion_rate,exploratory_cluster,anomaly_score,review_position,has_model_results,low_booking_count,exclusion_reason
49,b_143,25,19,0.7600,0.6316,2,0.5910,11,True,False,NaN
76,b_168,13,0,0.0000,NaN,<NA>,NaN,<NA>,False,True,nonpositive_bookings; missing_or_nonfinite_input
91,b_181,4862,556,0.1144,0.3885,0,0.6773,1,True,False,NaN
162,b_26,8,0,0.0000,NaN,<NA>,NaN,<NA>,False,True,nonpositive_bookings; missing_or_nonfinite_input
182,b_44,2,0,0.0000,NaN,<NA>,NaN,<NA>,False,True,nonpositive_bookings; missing_or_nonfinite_input
243,w_0,3460,776,0.2243,0.4768,0,0.5891,12,True,False,NaN


## 7.3 — Compare Demand and Booking Outcomes

Create separate views for:
- highest observed search volume;
- lowest observed booking conversion among records with ≥10 bookings.

Show counts alongside rates, cluster assignments, and anomaly positions.
The 10-booking cutoff is an exploratory display filter, not a validated
reliability threshold.

Keep w_0 separate from the comparison tables because its geographic
meaning remains unresolved. No operational priority score is assigned.

In [3]:
# Comparison population: preserve w_0 in the master profiles,
# but display it separately.
comparison_records = ward_profiles.loc[
    ~ward_profiles["is_w_0"]
].copy()

total_searches = ward_profiles["ward_searches"].sum()
assert total_searches > 0

ward_profiles["share_of_all_searches"] = (
    ward_profiles["ward_searches"] / total_searches
)

comparison_records["share_of_all_searches"] = (
    comparison_records["ward_searches"] / total_searches
)

DISPLAY_COLS = [
    "ward_num",
    "ward_searches",
    "share_of_all_searches",
    "ward_bookings",
    "ward_completed_rides",
    "ward_search_to_booking_rate",
    "ward_booking_completion_rate",
    "low_booking_count",
    "exploratory_cluster",
    "review_position",
]

highest_demand = comparison_records.sort_values(
    ["ward_searches", "ward_num"],
    ascending=[False, True],
).head(10)

lowest_conversion = (
    comparison_records.loc[
        comparison_records["ward_bookings"].ge(10)
    ]
    .sort_values(
        ["ward_search_to_booking_rate", "ward_searches", "ward_num"],
        ascending=[True, False, True],
    )
    .head(10)
)

def display_comparison(table):
    readable = table[DISPLAY_COLS].copy()

    for column in [
        "share_of_all_searches",
        "ward_search_to_booking_rate",
        "ward_booking_completion_rate",
    ]:
        readable[column + "_pct"] = readable.pop(column) * 100

    display(readable.round(2))

print("Highest search volume — w_0 displayed separately:")
display_comparison(highest_demand)

print("\nLowest observed conversion — ≥10 bookings, excluding w_0:")
display_comparison(lowest_conversion)

print("\nw_0 context:")
display_comparison(
    ward_profiles.loc[ward_profiles["is_w_0"]]
)

print(
    "\nSearch-share denominator includes all",
    len(ward_profiles),
    "source records, including w_0."
)

Highest search volume — w_0 displayed separately:


,ward_num,ward_searches,ward_bookings,ward_completed_rides,low_booking_count,exploratory_cluster,review_position,share_of_all_searches_pct,ward_search_to_booking_rate_pct,ward_booking_completion_rate_pct
91,b_181,4862,556,216,False,0,1,13.46,11.44,38.85
97,b_187,1874,440,141,False,0,10,5.19,23.48,32.05
38,b_133,1319,389,156,False,0,36,3.65,29.49,40.10
87,b_178,1119,279,129,False,0,43,3.10,24.93,46.24
39,b_134,1021,275,99,False,0,38,2.83,26.93,36.00
96,b_186,1008,266,122,False,0,51,2.79,26.39,45.86
25,b_121,823,224,108,False,0,55,2.28,27.22,48.21
234,b_91,643,150,72,False,0,57,1.78,23.33,48.00
36,b_131,516,69,29,False,0,16,1.43,13.37,42.03
19,b_116,472,191,85,False,0,54,1.31,40.47,44.50



Lowest observed conversion — ≥10 bookings, excluding w_0:


,ward_num,ward_searches,ward_bookings,ward_completed_rides,low_booking_count,exploratory_cluster,review_position,share_of_all_searches_pct,ward_search_to_booking_rate_pct,ward_booking_completion_rate_pct
91,b_181,4862,556,216,False,0,1,13.46,11.44,38.85
36,b_131,516,69,29,False,0,16,1.43,13.37,42.03
204,b_64,62,11,8,False,1,53,0.17,17.74,72.73
88,b_179,232,42,29,False,0,33,0.64,18.10,69.05
171,b_34,96,18,9,False,3,100,0.27,18.75,50.00
98,b_188,136,26,10,False,3,66,0.38,19.12,38.46
85,b_176,79,16,7,False,3,121,0.22,20.25,43.75
92,b_182,75,16,7,False,3,136,0.21,21.33,43.75
84,b_175,58,13,10,False,1,63,0.16,22.41,76.92
234,b_91,643,150,72,False,0,57,1.78,23.33,48.00



w_0 context:


,ward_num,ward_searches,ward_bookings,ward_completed_rides,low_booking_count,exploratory_cluster,review_position,share_of_all_searches_pct,ward_search_to_booking_rate_pct,ward_booking_completion_rate_pct
243,w_0,3460,776,370,False,0,12,9.58,22.43,47.68



Search-share denominator includes all 244 source records, including w_0.


**7.4 — Findings and Scope Limitations**

**Consolidated profiles**

Preserved all 244 source identifiers through validated one-to-one joins.
Shared modelling inputs match across the checked exports.

There are 241 records with cluster assignments and anomaly rankings.
The three zero-booking exclusions retain missing model results and
their exclusion reasons.

**Observed comparisons**

- b_181 has the highest recorded search volume: 4,862 searches,
  or 13.46% of searches across all 244 records.
  Its booking conversion is 11.44%, with 556 bookings.
- b_131 has 516 searches and 69 bookings, with 13.37% conversion.
  It appears second in the low-conversion view restricted to
  records with at least 10 bookings and excluding w_0.
- w_0 contributes 9.58% of recorded searches and is displayed separately
  because its geographic meaning remains unresolved.
- All ten highest-demand comparison records belong to exploratory
  cluster 0. Demand was a clustering input, so this is not independent
  validation of the clusters.

**Interpretation**

These tables describe observed demand and booking outcomes.
They do not establish driver shortages, causes, expected improvements,
or operational priorities.

The ≥10-bookings filter is exploratory and does not guarantee
rate reliability. Anomaly positions indicate unusualness, not severity.

**Spatial work remains pending**

No verified geographic lookup was identified in the inspected files.
All identifier-to-geography mappings remain unverified.

Maps, adjacency analysis, and geographic hotspot claims require
a documented identifier lookup and compatible boundary data.
The ward-level profile component is implemented; the spatial
component of Phase 7 remains pending.

## 7.5 — Export and Verify Ward-Level Outputs

Save the consolidated profiles and comparison views.
Record input hashes, software versions, and interpretation limits.

Reload the profile CSV to verify identifiers, counts, and missing
model outputs before downloading the checkpoint.

In [4]:
import hashlib
import json
import platform
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

WARD_REPORT_DIR = PROJECT_DIR / "reports" / "ward_level_intelligence"
WARD_REPORT_DIR.mkdir(parents=True, exist_ok=True)

export_tables = {
    "ward_profiles.csv": ward_profiles,
    "highest_demand_top10.csv": highest_demand,
    "lowest_conversion_top10.csv": lowest_conversion,
    "w_0_context.csv": ward_profiles.loc[ward_profiles["is_w_0"]],
}

for filename, table in export_tables.items():
    table.to_csv(WARD_REPORT_DIR / filename, index=False)

metadata = {
    "roadmap_phase": 7,
    "ward_profile_component": "implemented",
    "spatial_component": "pending_verified_geographic_mapping",
    "total_records": len(ward_profiles),
    "records_with_model_results": int(
        ward_profiles["has_model_results"].sum()
    ),
    "comparison_rules": {
        "highest_demand": "Top 10 by searches, excluding w_0",
        "lowest_conversion": (
            "Lowest 10 conversion rates among records with "
            "at least 10 bookings, excluding w_0"
        ),
        "search_share_denominator": (
            "Searches across all 244 source records, including w_0"
        ),
        "tie_breaking": (
            "Demand: ward_num ascending; conversion: searches descending, "
            "then ward_num ascending"
        ),
    },
    "input_sha256": {
        name: hashlib.sha256(path.read_bytes()).hexdigest()
        for name, path in INPUT_PATHS.items()
    },
    "versions": {
        "python": platform.python_version(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
    },
}

metadata_path = WARD_REPORT_DIR / "metadata.json"
metadata_path.write_text(
    json.dumps(metadata, indent=2), encoding="utf-8"
)

readme_path = WARD_REPORT_DIR / "README.md"
readme_path.write_text(
    """# Ward-Level Intelligence

Consolidated profiles preserve all 244 source identifiers.
Cluster assignments and anomaly rankings are available for 241 records.
Three zero-booking exclusions retain missing model results.

Rates and search shares in CSV files are fractions, not percentages.
review_position is the initial Isolation Forest review ordering.
exploratory_cluster is a descriptive cluster ID, not a priority category.
low_booking_count means fewer than 10 bookings and is a context flag.

Comparison tables exclude w_0; search shares include all source records.
The 10-booking display filter is exploratory, not a reliability guarantee.
Geographic mapping remains unverified for every identifier.
Driver-supply timing compatibility remains unresolved.
No operational intervention or causal conclusion is established.

The ward-level profile component is implemented.
The spatial component of Phase 7 remains pending a verified lookup
and compatible geographic boundaries.
""",
    encoding="utf-8",
)

# Verify CSV reload without replacing the in-memory profiles.
reloaded = pd.read_csv(WARD_REPORT_DIR / "ward_profiles.csv")

assert len(reloaded) == len(ward_profiles)
assert reloaded["ward_num"].is_unique
assert set(reloaded["ward_num"]) == feature_ids

reloaded_excluded = reloaded["ward_num"].isin(excluded_ids)
assert reloaded.loc[
    reloaded_excluded,
    ["exploratory_cluster", "anomaly_score", "review_position"],
].isna().all().all()

for column in [
    "ward_searches", "ward_bookings", "ward_completed_rides",
    "ward_search_to_booking_rate", "ward_booking_completion_rate",
    "anomaly_score", "share_of_all_searches",
]:
    np.testing.assert_allclose(
        reloaded[column],
        ward_profiles[column].to_numpy(),
        equal_nan=True,
        rtol=1e-10,
        atol=1e-12,
    )

print("Profile CSV reload checks passed.")

# Package only the outputs created by this cell.
output_paths = [
    WARD_REPORT_DIR / filename for filename in export_tables
] + [metadata_path, readme_path]

ZIP_PATH = Path("/content/NammaFlow_ward_intelligence_checkpoint.zip")

with ZipFile(ZIP_PATH, "w", compression=ZIP_DEFLATED) as archive:
    for path in output_paths:
        archive.write(path, arcname=str(path.relative_to(PROJECT_DIR)))

with ZipFile(ZIP_PATH) as archive:
    assert archive.testzip() is None
    print(f"Verified archive: {len(archive.namelist())} files")
    for name in archive.namelist():
        print(" -", name)

files.download(str(ZIP_PATH))

Profile CSV reload checks passed.
Verified archive: 6 files
 - reports/ward_level_intelligence/ward_profiles.csv
 - reports/ward_level_intelligence/highest_demand_top10.csv
 - reports/ward_level_intelligence/lowest_conversion_top10.csv
 - reports/ward_level_intelligence/w_0_context.csv
 - reports/ward_level_intelligence/metadata.json
 - reports/ward_level_intelligence/README.md


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>